# OneClick-AI-Cover — Kaggle RVC Training Preflight (experimental)

This notebook checks Kaggle's free GPU environment, locates voice recordings attached as a **private Kaggle Dataset**, and checks out the exact RVC revision pinned by this project. It is a guarded preflight, **not** a claim that end-to-end training has been automated or tested.

## Privacy and cost guardrails
- Use only your own voice or recordings you have explicit permission to use.
- Keep the Kaggle Dataset private. Never publish recordings, checkpoints, or models.
- Do not create public Gradio/tunnel links for private voice data.
- Use only a currently available free GPU. If unavailable, stop; do not enable paid services.
- Review [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks), [GPU usage](https://www.kaggle.com/docs/efficient-gpu-usage), and [Kaggle Terms](https://www.kaggle.com/terms).
- Pinned upstream backend: [RVC commit](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/tree/81eed5e8f68b6bed1789f682fe78cdd324495afc).

In [ ]:
import os, platform, subprocess, sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except Exception as exc:
    print('PyTorch diagnostic unavailable:', repr(exc))

if not Path('/kaggle/input').exists():
    print('NOTE: Run this notebook on Kaggle Notebooks.')
elif not Path('/proc/driver/nvidia').exists():
    print('STOP: No NVIDIA driver detected. Select an available free GPU in Settings; do not use paid acceleration.')

## 1. Attach your private voice dataset
In Kaggle, choose **Add Input** and attach a private dataset containing permitted recordings. WAV is the safest format. This cell lists filenames and sizes only; it does not play or upload audio.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
AUDIO_EXTS = {'.wav', '.flac', '.mp3', '.m4a', '.ogg'}
audio_files = sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTS)
print('Audio files found:', len(audio_files))
for p in audio_files[:100]:
    print(f'{p} ({p.stat().st_size / (1024*1024):.2f} MiB)')
if not audio_files:
    print('No audio found. Attach your private dataset and rerun this cell.')

## 2. Check out the exact pinned RVC backend
This uses the revision pinned by OneClick-AI-Cover, not a moving `main` branch. It does not install packages or begin training.

In [ ]:
RVC_COMMIT = '81eed5e8f68b6bed1789f682fe78cdd324495afc'
RVC_DIR = Path('/kaggle/working/Retrieval-based-Voice-Conversion-WebUI')
if not RVC_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git', str(RVC_DIR)], check=True)
subprocess.run(['git', '-C', str(RVC_DIR), 'fetch', '--depth', '1', 'origin', RVC_COMMIT], check=True)
subprocess.run(['git', '-C', str(RVC_DIR), 'checkout', '--detach', RVC_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(RVC_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned revision:', actual)
assert actual == RVC_COMMIT, f'Unexpected RVC revision: {actual}'
print('Backend revision verified.')

## 3. Compatibility preflight
The pinned backend specifies Python 3.12 requirements. Dependency installation, pretrained-asset setup, and actual training must be tested against the current Kaggle image before they can safely be automated.

In [ ]:
print('Python 3.12:', sys.version_info[:2] == (3, 12))
for name in ['requirments_cu128_py312.txt', 'requirments_cu118_py312.txt', 'requirments_cpu_py312.txt']:
    print(name, 'exists:', (RVC_DIR / name).is_file())
if sys.version_info[:2] != (3, 12):
    print('STOP: Runtime does not match the pinned backend requirements.')
elif not torch.cuda.is_available():
    print('STOP: No free GPU is available. Do not switch to paid services.')
else:
    print('Basic environment checks passed; this is not yet proof that RVC training will run.')

## 4. Why this notebook stops before training
The upstream pipeline needs compatible pretrained assets (including HuBERT and F0/RMVPE resources), audio preprocessing, feature extraction, training, index generation, and model export. Simply cloning the backend or launching its WebUI is not proof of a working training pipeline.

**Not implemented yet:** verified private Kaggle training orchestration, automatic checkpoint export, and automatic model download/import. This notebook deliberately does not expose a public UI or pretend that these stages have passed. The next engineering milestone is to validate each stage in a controlled session, then add export only after success.

When you have an authorized inference-ready `.pth` model, download it privately and use OneClick-AI-Cover's **Import Voice Model** tab. Never commit voice data or model files to this public repository.